# Engine Pipeline Walkthrough

This notebook is a **training manual** for the EWIS Engine — the central orchestrator of the
Energy–Water Information System toolkit.  By the time you reach the final cell you will
understand every stage of the data flow and be ready to extend the engine for your own
use-cases.

The engine follows a five-stage pipeline: **ingest → normalize → enrich → score → export**.
Raw telemetry arrives as a plain Python dictionary, gets validated into a
`TelemetryPayload`, passes through user-registered plugins for enrichment, is scored
against a library of sustainability metrics, and finally lands in an `EWISReport` that
you can serialise to JSON, log, or feed into downstream dashboards.

The walkthrough is fully self-contained — every import, every object, and every assertion
lives right here so you can run the notebook top-to-bottom in a single kernel session
without any external setup beyond having the `ewis` package installed.

## Stage 1 — Ingest: Building a TelemetryPayload

`TelemetryPayload` is a **Pydantic v2 model** that defines the canonical shape of every
telemetry snapshot the engine can process.  It enforces type constraints, value ranges,
and cross-field invariants at construction time so that downstream code never has to
worry about malformed data.

**Required fields** — every payload must include these:

| Field | Type | Constraint |
|---|---|---|
| `datacenter_id` | `str` | min length 1 |
| `region` | `str` | min length 1 |
| `timestamp_utc` | `str` | min length 1 |
| `power_mw` | `float` | ≥ 0.0 |
| `it_load_mw` | `float` | ≥ 0.0, must be ≤ `power_mw` |
| `pue` | `float` | ≥ 1.0 |

**Optional fields** include grid context (`grid_capacity_mw`, `base_grid_load_mw`,
`price_usd_per_mwh`, `carbon_intensity_kgco2_per_mwh`), weather observations
(`ambient_temp_c`, `humidity_pct`, `wind_m_s`, `precip_mm`), and a free-form
`workload` dictionary for AI/ML metadata like token counts and energy consumption.

In [ ]:
from ewis.core.payloads import TelemetryPayload

raw_payload = {
    "datacenter_id": "DC-TRAIN-01",
    "region": "ERCOT",
    "timestamp_utc": "2025-06-15T14:30:00Z",
    "power_mw": 45.0,
    "it_load_mw": 36.0,
    "pue": 1.3,
    # Grid context
    "grid_capacity_mw": 500.0,
    "base_grid_load_mw": 350.0,
    "price_usd_per_mwh": 42.5,
    "carbon_intensity_kgco2_per_mwh": 410.0,
    # Weather observations
    "ambient_temp_c": 33.2,
    "humidity_pct": 62.0,
    "wind_m_s": 3.1,
    "precip_mm": 0.0,
    # Workload metadata
    "workload": {
        "model": "gpt-4-turbo",
        "tokens": 1_500_000,
        "energy_kwh": 320.0,
    },
}

payload = TelemetryPayload(**raw_payload)
print(payload)

## Stage 2 — Normalize: Validation & Serialization

When you construct a `TelemetryPayload`, Pydantic validates every field against its
declared type and constraints.  If anything is out of range — a negative `power_mw`, a
`pue` below 1.0, or an `it_load_mw` that exceeds `power_mw` — construction fails
immediately with a descriptive `ValidationError`.

Once validated, call **`to_dict()`** to obtain a clean Python dictionary suitable for
the rest of the pipeline.  Under the hood this calls Pydantic's `model_dump()` with
`exclude_none=True`, so optional fields that were not supplied are omitted rather than
appearing as `None`.  This keeps downstream metric functions simple — they only need to
check for key existence, not for `None` sentinels.

In [ ]:
import json

validated = payload.to_dict()
print(json.dumps(validated, indent=2, default=str))

### Handling Validation Errors

Pydantic raises a `ValidationError` the moment it encounters data that violates the
schema.  This is *by design* — it is far better to reject bad telemetry at the front
door than to let it silently corrupt downstream metrics.

Below we trigger two common validation failures:

1. **`it_load_mw` exceeds `power_mw`** — a physical impossibility (the IT load cannot
   draw more power than the facility supplies).
2. **`pue` below 1.0** — PUE is defined as total facility power divided by IT load, so
   it can never be less than 1.0.

Each error message pinpoints exactly which field failed and why, making it easy to fix
upstream data quality issues.

In [ ]:
from pydantic import ValidationError

# Error 1: it_load_mw exceeds power_mw
print("--- Error 1: it_load_mw > power_mw ---")
try:
    TelemetryPayload(
        datacenter_id="DC-BAD-01",
        region="ERCOT",
        timestamp_utc="2025-06-15T14:30:00Z",
        power_mw=40.0,
        it_load_mw=50.0,  # 50 > 40 → invalid
        pue=1.3,
    )
except ValidationError as exc:
    print(exc)

# Error 2: pue below 1.0
print("\n--- Error 2: pue < 1.0 ---")
try:
    TelemetryPayload(
        datacenter_id="DC-BAD-02",
        region="ERCOT",
        timestamp_utc="2025-06-15T14:30:00Z",
        power_mw=45.0,
        it_load_mw=36.0,
        pue=0.8,  # below 1.0 → invalid
    )
except ValidationError as exc:
    print(exc)

## Stage 3 — Enrich: Registering & Running Plugins

Plugins are the primary extension mechanism of the EWIS Engine.  Any class that inherits
from `BasePlugin` and implements `execute(payload, context)` can be registered with the
engine before a run.

The plugin lifecycle works as follows:

1. **Registration** — call `engine.register(plugin)`.  The engine checks for duplicate
   names, then calls `plugin.initialize(context)` so the plugin can read configuration
   or warm caches.
2. **Execution** — during `engine.run()`, the `PluginManager` iterates through all
   registered plugins in insertion order, calling `plugin.execute(payload, context)` on
   each.  Every plugin returns a `PluginResult` (name, ok, data, metadata).
3. **Error handling** — if any plugin raises an exception the manager wraps it in a
   `PluginExecutionError` that identifies which plugin failed.

Let's create an engine and register the built-in **CoolingOptimizerPlugin**, which
estimates a recommended cooling posture based on ambient temperature, humidity, and PUE.

In [ ]:
from ewis.core.engine import EWISEngine, EngineConfig
from ewis.plugins.builtin.cooling_optimizer import CoolingOptimizerPlugin

engine = EWISEngine()  # default EngineConfig
engine.register(CoolingOptimizerPlugin())

print("Registered plugins:", list(engine.plugins.plugins.keys()))

## Stage 4 — Score: Running the Engine

Calling `engine.run(raw_dict)` triggers the full pipeline in a single method:

1. The raw dictionary is validated into a `TelemetryPayload` and serialised back to a
   clean dict via `to_dict()`.
2. All registered plugins are executed against the validated payload.  Each returns a
   `PluginResult`.
3. The built-in metric functions — `grid_stress_index`, `energy_per_token`, and
   `carbon_per_token` — are computed over the validated payload.
4. Everything is bundled into an `EWISReport` and returned.

Notice that `engine.run()` accepts the *raw* dictionary, not a pre-built
`TelemetryPayload`.  This keeps the public API simple — callers never need to import
the payload model directly.

In [ ]:
report = engine.run(raw_payload)

print("Return type:", type(report))
print("Report fields:", list(vars(report).keys()))

## Stage 5 — Export: Inspecting the Report

The `EWISReport` dataclass offers two serialisation helpers:

| Method | What it returns |
|---|---|
| `summary()` | A compact dictionary with the datacenter id, region, timestamp, all metrics, and a simplified view of plugin results (name, ok, metadata — **no raw data**). Ideal for dashboards or log lines. |
| `to_json_dict()` | The *full* report including the complete validated payload, all metrics, and every plugin result with its `data` payload. Use this for archival or inter-service communication. |

Let's look at both.

In [ ]:
print("=== report.summary() ===")
print(json.dumps(report.summary(), indent=2, default=str))

print("\n=== report.to_json_dict() keys ===")
print(list(report.to_json_dict().keys()))

### Accessing Individual Metrics

Every metric lives in `report.metrics` keyed by its function name.  Each value is a
dictionary with at least `"value"` (the numeric result, or `None` when inputs are
missing) and `"notes"` (contextual information or error details).

| Metric | What it measures |
|---|---|
| **`grid_stress_index`** | How much stress the datacenter places on the local power grid. Combines the ratio of facility power to available grid headroom with a weather-derived severity factor. Returns `None` when `grid_capacity_mw` or `base_grid_load_mw` is absent. |
| **`energy_per_token`** | Energy efficiency of the AI workload in kWh per token.  Computed as `workload.energy_kwh / workload.tokens`. |
| **`carbon_per_token`** | Carbon footprint per token in kg CO₂.  Multiplies the grid carbon intensity by energy consumed, then divides by the token count. |

In [ ]:
gsi = report.metrics["grid_stress_index"]
print(f"Grid Stress Index : {gsi['value']:.4f}")
print(f"  Components       : {gsi.get('components', {})}")

ept = report.metrics["energy_per_token"]
print(f"\nEnergy per Token  : {ept['value']:.6f} kWh/token")

cpt = report.metrics["carbon_per_token"]
print(f"Carbon per Token  : {cpt['value']:.6f} kg CO\u2082/token")

### Plugin Results

Each plugin returns a frozen `PluginResult` dataclass with four fields:

- **`name`** — the unique plugin identifier (matches `plugin.name`).
- **`ok`** — `True` if the plugin executed without error.
- **`data`** — the plugin's primary output dictionary.
- **`metadata`** — auxiliary information such as the PUE value that influenced the result.

In [ ]:
for result in report.plugin_results:
    print(f"Plugin : {result.name}")
    print(f"  ok   : {result.ok}")
    print(f"  data : {result.data}")
    print(f"  meta : {result.metadata}")
    print()

## Customizing the Engine Configuration

The `EngineConfig` dataclass controls engine-wide behaviour.  Pass it to the
`EWISEngine` constructor to override the defaults.

| Field | Type | Default | Purpose |
|---|---|---|---|
| `allow_side_effects` | `bool` | `False` | When `True`, plugins are permitted to perform side effects such as writing files or calling external APIs.  The flag is forwarded to `PluginContext.allow_side_effects`. |
| `plugin_timeout_s` | `int` | `10` | Maximum number of seconds a single plugin is allowed to run before the engine should consider it timed out. |
| `extra` | `dict \| None` | `None` | An arbitrary dictionary that is passed through to `PluginContext.config`.  Use it to inject feature flags, API keys, or environment-specific settings that plugins can read via `context.get(key)`. |

In [ ]:
custom_config = EngineConfig(
    allow_side_effects=True,
    plugin_timeout_s=30,
    extra={"custom_key": "value"},
)

custom_engine = EWISEngine(config=custom_config)

print("context.config             :", custom_engine.context.config)
print("context.allow_side_effects :", custom_engine.context.allow_side_effects)
print("context.plugin_timeout_s   :", custom_engine.context.plugin_timeout_s)

## Summary

You have now traced a telemetry snapshot through every stage of the EWIS Engine:

1. **Ingest** — raw telemetry arrives as a plain Python dictionary with datacenter,
   grid, weather, and workload fields.
2. **Normalize** — `TelemetryPayload` validates types, ranges, and cross-field
   invariants, then `to_dict()` produces a clean dictionary with no `None` values.
3. **Enrich** — registered plugins (e.g., `CoolingOptimizerPlugin`) augment the
   payload with domain-specific insights such as cooling posture recommendations.
4. **Score** — built-in metric functions compute `grid_stress_index`,
   `energy_per_token`, and `carbon_per_token` from the validated payload.
5. **Export** — the `EWISReport` bundles everything together and offers `summary()`
   for dashboards and `to_json_dict()` for full archival.

**Next steps:**

- Write a custom plugin by subclassing `BasePlugin` and implementing `execute()`.
- Experiment with different `EngineConfig.extra` values to control plugin behaviour.
- Explore the companion notebooks (`01`–`03`) for deep-dives into individual metrics
  and plugins.
- Integrate the engine into a scheduled pipeline that ingests live telemetry from your
  datacenter monitoring stack.